# C3 / C4 — GPU-добор для статьи (Lightning AI)

Обучает две оставшиеся глубокие конфигурации — `C3_pure_cnn` и
`C4_pure_transformer` — на GPU. Тот же код и те же данные, что и в основном
прогоне; отличается только устройство, а оно на метрику не влияет.

## Как запустить в Lightning AI (бесплатно, 22 GPU-ч/мес)
1. Зайди на **lightning.ai** → создай **New Studio** (или открой существующую).
2. Перетащи этот `.ipynb` в файловый браузер слева и открой его.
3. Справа выбери **GPU-машину** (хватит **L4**). Дождись переключения Studio
   на GPU — вверху появится имя карты.
4. Перетащи в тот же файловый браузер свой **`kaggle.json`**
   (Kaggle → аватар → Settings → API → *Create New Token*).
5. **Run all**. В конце ноутбук напечатает блок между `RESULTS_JSON_START` и
   `RESULTS_JSON_END` — **скопируй его целиком обратно в чат Claude**.

Время: ~1–2 часа на конфигурацию (итого ~2–4 ч). Ноутбук так же работает в
Colab — там GPU включается через *Runtime → Change runtime type → GPU*.

In [ ]:
# 1. Проверка, что GPU действительно выделен
import torch
assert torch.cuda.is_available(), (
    "GPU не выделен! В Lightning выбери GPU-машину справа (L4); "
    "в Colab: Runtime -> Change runtime type -> GPU. Затем запусти заново.")
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

In [ ]:
# 2. Публичный код проекта + зависимости (torch на GPU-машине уже с CUDA)
!git clone --depth 1 -b v2-restructure https://github.com/SergeySolovyev/smart-contract-vuln-detection-from-bytecode.git repo
!pip -q install "xgboost" "scikit-learn" pyarrow joblib kaggle
import os
print("dl_pipeline на месте:", os.path.exists("repo/src/dl_pipeline.py"))

In [ ]:
# 3. Данные: два parquet из твоего приватного Kaggle-датасета по твоему kaggle.json.
import os, glob, zipfile, shutil, pathlib
home_kaggle = pathlib.Path.home() / ".kaggle" / "kaggle.json"
try:
    # Colab: интерактивный виджет загрузки
    from google.colab import files
    print("Загрузи kaggle.json:")
    up = files.upload()
    home_kaggle.parent.mkdir(parents=True, exist_ok=True)
    home_kaggle.write_bytes(list(up.values())[0])
except ImportError:
    # Lightning / любой Jupyter: подхватываем kaggle.json из рабочей папки
    if not home_kaggle.exists():
        cand = next((p for p in [
            "kaggle.json",
            os.path.expanduser("~/kaggle.json"),
            "/teamspace/studios/this_studio/kaggle.json",
        ] if os.path.exists(p)), None)
        assert cand, ("Перетащи kaggle.json в файловый браузер Studio "
                      "(в рабочую папку), затем запусти эту ячейку снова.")
        home_kaggle.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(cand, home_kaggle)
os.chmod(home_kaggle, 0o600)
print("kaggle.json готов:", home_kaggle)

DS = "sergeisolovyev/defi-bytecode-features-v2"
for f in ("train_v2.parquet", "test_v2.parquet"):
    !kaggle datasets download -d {DS} -f {f} -p data_v2
for z in glob.glob("data_v2/*.zip"):
    zipfile.ZipFile(z).extractall("data_v2")
print("data_v2:", os.listdir("data_v2"))

In [ ]:
# 4. Обучение C3 и C4 на GPU (тот же вызов, что в основном пайплайне)
import sys, time, json, pathlib
import numpy as np, pandas as pd, torch
sys.path.insert(0, "repo/src")
from dl_pipeline import (BytecodeTokenizer, build_token_ids,
                         DL_EXPERIMENTS, run_dl_experiment)

LABELS = ["access-control", "arithmetic", "bad-randomness", "double-spending",
          "locked-ether", "other", "reentrancy", "unchecked-calls"]
FEATURES = json.load(open("repo/data/feature_columns.json"))
WANT = {"C3_pure_cnn", "C4_pure_transformer"}

# Guards: these two configs must be comparable with the 8 already trained on
# Kaggle, so the inputs have to be IDENTICAL, not merely similar. Verified
# against the Kaggle trainer: same 67 features in the same order, tokenizer
# fit on exactly the first 20000 train rows -> vocabulary of 498 tokens.
# If either differs, stop loudly rather than emit numbers that silently
# cannot be compared with the rest of the ablation.
assert len(FEATURES) == 67, f"expected 67 features, got {len(FEATURES)}"

cols = ["bytecode"] + FEATURES + LABELS
tr = pd.read_parquet("data_v2/train_v2.parquet", columns=cols)
te = pd.read_parquet("data_v2/test_v2.parquet", columns=cols)
print(f"train {len(tr):,}  test {len(te):,}  device=cuda")
assert len(tr) == 89973 and len(te) == 11247, "unexpected split sizes"

tok = BytecodeTokenizer().fit(tr["bytecode"].head(20000))
print("vocab:", tok.vocab_size)
assert tok.vocab_size == 498, (
    f"tokenizer vocab {tok.vocab_size} != 498 expected -- inputs differ from "
    "the Kaggle run, results would NOT be comparable. Stop and report this.")
tr_ids = build_token_ids(tr["bytecode"], tok)
te_ids = build_token_ids(te["bytecode"], tok)
y_tr = tr[LABELS].to_numpy("float32"); y_te = te[LABELS].to_numpy("float32")
X_tr = tr[FEATURES].to_numpy("float32"); X_te = te[FEATURES].to_numpy("float32")
del tr, te

RUNS = pathlib.Path("runs_out"); RUNS.mkdir(exist_ok=True)
out = {}
for cfg in DL_EXPERIMENTS:
    if cfg["name"] not in WANT:
        continue
    print(f"\n=== {cfg['name']} ===")
    t0 = time.time()
    res = run_dl_experiment(cfg, tok, tr_ids, y_tr, X_tr, te_ids, y_te, X_te,
                            runs_dir=RUNS, wandb_run=None)
    mf = res.get("macro_f1_external", res.get("macro_f1"))
    print(f"  {cfg['name']} macro_f1={mf:.4f}  ({(time.time()-t0)/60:.0f} min)")
    out[cfg["name"]] = res
print("\nобе конфигурации обучены")

In [ ]:
# 5. Отдать результат: JSON для копипаста в чат + сохранённые файлы
import json
for name, res in out.items():
    open(f"{name}.json", "w").write(json.dumps(res))
print("\n=========== СКОПИРУЙ ВСЁ МЕЖДУ МЕТКАМИ ОБРАТНО В ЧАТ CLAUDE ===========\n")
print("RESULTS_JSON_START")
print(json.dumps(out))
print("RESULTS_JSON_END")
print("\nТакже сохранены файлы:", [f"{n}.json" for n in out],
      "\n(их можно скачать из файлового браузера Studio, но хватит и JSON выше)")